In [6]:
import numpy  as np
import pandas as pd

pd.options.mode.chained_assignment = None  # default='warn'

# Tester la qualité des prédiction

+ Divisé le dataframe :
    * 'train_df' avec 6000 lignes dr 'listings' ;
    * 'test_df' avec le reste.
* Modifier la fonction, changer la datagrame 'temp_df'. Idem avec le dataframe 'listings' à 'train_df' pour que seul le dataset de training soit utilisé ;
* Utiliser la méthode '`Series apply`' pour appliquer la fonction 'predict_price' sur les valeurs de la colonne 'accomodates' du dataframe de test ;
* Assigner l'objet '`Series`' résultant à la colonnes 'predicted_price' de 'test_df'.

In [5]:
listings          = pd.read_csv('paris_airbnb.csv')
stripped_commas   = listings['price'].str.replace(',', '')
stripped_dollars  = stripped_commas.str.replace('$', '')
listings['price'] = stripped_dollars.astype('float')

In [4]:
train_df = listings.iloc[:6000]
test_df  = listings.iloc[6000:]

In [3]:
# 80% train / 20% test
train_df = listings.sample(frac=0.75, random_state=42)
test_df  = listings.drop(train_df.index)

print(f"Il y a {len(train_df)} logements dans le jeu d'entraînement \net {len(test_df)} logements dans le jeu de test.")

Il y a 6000 logements dans le jeu d'entraînement 
et 2000 logements dans le jeu de test.


In [7]:
def predict_price(new_listing):
    temp_df             = train_df.copy()
    temp_df['distance'] = temp_df['accommodates'].apply(lambda x: np.abs(x - new_listing))
    temp_df             = temp_df.sort_values('distance')
    nearest_neighbors   = temp_df.iloc[0:5]['price']
    predicted_price     = nearest_neighbors.mean()
    return(predicted_price)

In [8]:
test_df['predicted_price'] = test_df['accommodates'].apply(lambda x: predict_price(x))

### Les métriques d'erreur

* Utiliser la méthode '`numpy.absolute()`' pour calculer l'erreur absolue moyenne MAE entre 'predicted_price' et 'price' ;
* Assigner le résultat à la variable MAE.

$$
\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} \left| y_i - \hat{y}_i \right|
= \frac{|y_1 - \hat{y}_1| + |y_2 - \hat{y}_2| + \cdots + |y_n - \hat{y}_n|}{n}
$$

In [9]:
test_df.head()

,host_response_rate,host_acceptance_rate,host_listings_count,latitude,longitude,city,zipcode,state,accommodates,room_type,bedrooms,bathrooms,beds,price,cleaning_fee,security_deposit,minimum_nights,maximum_nights,number_of_reviews,predicted_price
7505,NaN,NaN,1.0,48.87057,2.36713,Paris,NaN,Île-de-France,2,Entire home/apt,0.0,1.0,1.0,30.0,NaN,NaN,1,1125,1,101.2
6917,NaN,NaN,1.0,48.89175,2.33836,Paris,75018,Île-de-France,2,Entire home/apt,1.0,1.0,1.0,65.0,NaN,$300.00,1,1125,0,101.2
4218,100%,NaN,2.0,48.87874,2.36133,Paris,75010,Île-de-France,2,Entire home/apt,0.0,1.0,1.0,90.0,$25.00,$400.00,3,15,374,101.2
7808,NaN,NaN,1.0,48.84095,2.31350,Paris,75015,Île-de-France,4,Entire home/apt,1.0,1.0,2.0,100.0,NaN,NaN,4,1125,2,113.2
5093,NaN,NaN,1.0,48.87864,2.39705,Paris,75019,Île-de-France,2,Entire home/apt,0.0,1.0,1.0,35.0,$12.00,$150.00,14,60,71,101.2


In [10]:
test_df['error'] = np.abs(test_df['predicted_price'] - test_df['price'])
mae = test_df['error'].mean()
print(f"Le MAE du modèle est de {mae:.2f} $.")

Le MAE du modèle est de 52.17 $.
